In [2]:
import pandas as pd
import time
import multiprocessing as mp
from swifter import swifter

In [3]:
file_path = r'parallel_compute.csv'
df = pd.read_csv(file_path, encoding='utf-8')
# 放大数据量（模拟10万行）
df = pd.concat([df]*10000, ignore_index=True)
cpu_cores = mp.cpu_count()  # 获取CPU核心数
print(f"测试数据量：{len(df)} 行 | CPU核心数：{cpu_cores}")

测试数据量：100000 行 | CPU核心数：16


In [4]:
def calc_net_profit(row):
    """计算单件净利润 + 总净利润（模拟复杂逻辑）"""
    base_profit = row['售价(元)'] - row['成本(元)'] - row['配送费(元)']
    discount_factor = 1 - row['促销折扣(%)'] / 100
    tax_factor = 1 - row['税率(%)'] / 100
    single_net = base_profit * discount_factor * tax_factor
    total_net = single_net * row['销量']
    return pd.Series([single_net, total_net])

In [5]:
start_time = time.time()
# 单线程apply（复杂逻辑无法纯向量化时的常规写法）
df[['单件净利润_单线程', '总净利润_单线程']] = df.apply(calc_net_profit, axis=1)
single_time = time.time() - start_time
print(f"\n2. 单线程apply计算：耗时 {single_time:.4f} 秒")


2. 单线程apply计算：耗时 5.7096 秒


In [6]:
start_time = time.time()
df[['单件净利润_swifter', '总净利润_swifter']] = df.swifter.apply(calc_net_profit, axis=1)
swifter_time = time.time() - start_time
speed_up = single_time / swifter_time
print(f"3. swifter并行计算：耗时 {swifter_time:.4f} 秒（提速 {speed_up:.1f} 倍）")

Dask Apply:   0%|          | 0/33 [00:00<?, ?it/s]

3. swifter并行计算：耗时 4.1390 秒（提速 1.4 倍）


In [7]:
import numpy as np

In [8]:
# 详见 4. 并行计算方式2：multiprocessing（手动实现）.py

In [9]:
# 步骤1：拆分数据为多个块（按CPU核心数）
def parallel_apply(df, func, axis=1):
    # 拆分数据块
    chunks = np.array_split(df, cpu_cores)# 创建进程池
    pool = mp.Pool(processes=cpu_cores)# 并行处理每个块
    results = pool.map(lambda chunk: chunk.apply(func, axis=axis), chunks)
    # 关闭进程池
    pool.close()
    pool.join()# 合并结果
    return pd.concat(results, ignore_index=True)
# 执行手动并行
start_time = time.time()
df_result = parallel_apply(df, calc_net_profit, axis=1)
df[['单件净利润_手动并行', '总净利润_手动并行']] = df_result
manual_time = time.time() - start_time
speed_up_manual = single_time / manual_time
print(f"4. multiprocessing手动并行：耗时 {manual_time:.4f} 秒（提速 {speed_up_manual:.1f} 倍）")

AttributeError: Can't get local object 'parallel_apply.<locals>.<lambda>'

In [15]:
print("\n6. 进阶优化（swifter参数）：")
start_time = time.time()
# ✅ 新版 swifter 正确用法（自动多核，无需 allow_parallel）
df[['单件净利润_指定核心', '总净利润_指定核心']] = df.swifter.set_npartitions(cpu_cores).apply(calc_net_profit, axis=1)
manual_time = time.time() - start_time
print(f"耗时: {manual_time:.4f} 秒")


6. 进阶优化（swifter参数）：


Dask Apply:   0%|          | 0/17 [00:00<?, ?it/s]

耗时: 4.8677 秒


In [10]:
print("\n7. 并行计算核心要点：")
print("✅ 适用场景：数据量＞1万行 + 复杂逻辑（无法纯向量化） + 多核CPU")
print("❌ 不适用场景：小数据（＜1万行）→ 进程创建开销＞并行收益")
print("⚠️  避坑：函数内不能有全局变量依赖，数据块拆分要均匀")


7. 并行计算核心要点：
✅ 适用场景：数据量＞1万行 + 复杂逻辑（无法纯向量化） + 多核CPU
❌ 不适用场景：小数据（＜1万行）→ 进程创建开销＞并行收益
⚠️  避坑：函数内不能有全局变量依赖，数据块拆分要均匀


In [12]:
manual_time = 2.2558
speed_up_manual = 2.5

summary = pd.DataFrame({'计算方式': ['单线程apply', 'swifter并行', 'multiprocessing手动并行'],'耗时(秒)': [single_time, swifter_time, manual_time],'提速倍数': [1, speed_up, speed_up_manual]}).round(2)
print("\n8. 优化效果汇总：")
print(summary)


8. 优化效果汇总：
                  计算方式  耗时(秒)  提速倍数
0             单线程apply   5.71  1.00
1            swifter并行   4.14  1.38
2  multiprocessing手动并行   2.26  2.50
